# RealSaS Knight — Geppetto mechanically-meaningful V2 Run-All

Demo-only fresh Geppetto refit for the Knight witness. This run **does not optimize for a visually clean/minimal skeleton**. The teacher target is mechanically selected: skin-supported articulation + required hierarchy + root-motion anchor + terminal mechanical spans.

**PASS rule is frozen before training:** every one of the 4 diffusion seeds must pass the unchanged metric/topology gates at **3 consecutive 64-step checks**. No historical checkpoint. No teacher inputs at free-running inference. Product authority/generalization remain forbidden.

Execution code is pinned to commit `1cec9a7cbdbe84e22443ffe89df5934391aa5192` (self-hosted CI run `36164993541` passed before this notebook was sealed).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import gzip, hashlib, json, os, shutil, subprocess, sys

BASE_HANDOFF = Path('/content/drive/MyDrive/RealSaS_SUBJECT2_KNIGHT_DEMO_V2_20260924/GEPPETTO_STAGE27_HANDOFF')
V2_HANDOFF = Path('/content/drive/MyDrive/RealSaS_SUBJECT2_KNIGHT_DEMO_V2_20260924/GEPPETTO_STAGE27_MECHANICALLY_MEANINGFUL_V2_HANDOFF')
DRIVE_OUTPUT = Path('/content/drive/MyDrive/RealSaS_SUBJECT2_KNIGHT_DEMO_V2_20260924/GEPPETTO_STAGE27_MECHANICALLY_MEANINGFUL_V2_OUTPUT')
WORK = Path('/content/realsas_geppetto_mechanically_meaningful_v2')
SRC = WORK / 'source'
LOCAL_OUTPUT = WORK / 'output'
EXECUTION_COMMIT = '1cec9a7cbdbe84e22443ffe89df5934391aa5192'
EXPECTED_CI_RUN_ID = 36164993541

EXPECTED_TARGET_COUNT = 28
EXPECTED_ROLE_COUNTS = {
    'ROOT_MOTION_ANCHOR': 1,
    'SKIN_SUPPORTED_ARTICULATION': 20,
    'STRUCTURAL_BRIDGE': 0,
    'TERMINAL_EXTENSION_SOURCE': 2,
    'TERMINAL_TIP_SYNTHETIC': 5,
}
EXPECTED_TARGET_RULE = 'SKIN_SUPPORTED_PLUS_STRUCTURAL_BRIDGES_PLUS_ROOT_MOTION_ANCHOR_PLUS_TERMINAL_EXTENSIONS_AND_TIPS'
REQUIRED_CONSECUTIVE_FULL_PASSES = 3
CHECK_EVERY = 64
DIFFUSION_SEEDS = [11, 23, 47, 89]

def sha256(path: Path) -> str:
    h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1<<20), b''):
            h.update(chunk)
    return h.hexdigest()

# Verify the original sealed handoff that owns model/cameras/observations/surface.
base_manifest_path = BASE_HANDOFF / 'HANDOFF_MANIFEST.json'
assert base_manifest_path.is_file(), f'MISSING_BASE_HANDOFF_MANIFEST:{base_manifest_path}'
base_manifest=json.loads(base_manifest_path.read_text(encoding='utf-8'))
assert base_manifest['schema']=='RealSaS.KnightGeppettoStage27Handoff.v1', base_manifest
assert base_manifest['status']=='PASS', base_manifest
assert base_manifest['execution_class']=='DEMO_WITNESS', base_manifest
assert base_manifest['product_authority_claimed'] is False, base_manifest
for rel, meta in base_manifest['files'].items():
    p=BASE_HANDOFF/rel
    assert p.is_file(), f'BASE_HANDOFF_FILE_MISSING:{rel}'
    assert p.stat().st_size==int(meta['bytes']), f'BASE_HANDOFF_SIZE_DRIFT:{rel}'
    assert sha256(p)==meta['sha256'], f'BASE_HANDOFF_SHA_DRIFT:{rel}'

# Verify the new target/prereg/notebook handoff as one sealed unit.
v2_manifest_path = V2_HANDOFF / 'HANDOFF_MANIFEST_V2.json'
assert v2_manifest_path.is_file(), f'MISSING_V2_HANDOFF_MANIFEST:{v2_manifest_path}'
v2_manifest=json.loads(v2_manifest_path.read_text(encoding='utf-8'))
assert v2_manifest['schema']=='RealSaS.KnightGeppettoMechanicallyMeaningfulStage27Handoff.v2', v2_manifest
assert v2_manifest['status']=='PASS', v2_manifest
assert v2_manifest['execution_class']=='DEMO_WITNESS', v2_manifest
assert v2_manifest['execution_repo_commit']==EXECUTION_COMMIT, v2_manifest
assert int(v2_manifest['self_hosted_ci_run_id'])==EXPECTED_CI_RUN_ID, v2_manifest
assert int(v2_manifest['target_count'])==EXPECTED_TARGET_COUNT, v2_manifest
assert v2_manifest['target_role_counts']==EXPECTED_ROLE_COUNTS, v2_manifest
assert v2_manifest['target_rule']==EXPECTED_TARGET_RULE, v2_manifest
assert int(v2_manifest['required_consecutive_full_passes'])==REQUIRED_CONSECUTIVE_FULL_PASSES, v2_manifest
assert v2_manifest['product_authority_claimed'] is False, v2_manifest
assert v2_manifest['generalization_claimed'] is False, v2_manifest
for rel, meta in v2_manifest['files'].items():
    p=V2_HANDOFF/rel
    assert p.is_file(), f'V2_HANDOFF_FILE_MISSING:{rel}'
    assert p.stat().st_size==int(meta['bytes']), f'V2_HANDOFF_SIZE_DRIFT:{rel}'
    assert sha256(p)==meta['sha256'], f'V2_HANDOFF_SHA_DRIFT:{rel}'
print('GEPPETTO_MECHANICALLY_MEANINGFUL_V2_HANDOFF_VERIFIED', EXECUTION_COMMIT)


In [ ]:
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)
LOCAL_OUTPUT.mkdir(parents=True)
MATERIALIZED = WORK / 'materialized'
MATERIALIZED.mkdir(parents=True)

# Materialize the exact Stage15 rigging surface from the original sealed handoff.
compressed = dict(base_manifest.get('compressed_payloads') or {})
surface_transport = compressed.get('qualified_rigging_surface.json')
if surface_transport:
    assert surface_transport.get('compression') == 'gzip', surface_transport
    archive_rel = str(surface_transport['archive'])
    archive_path = BASE_HANDOFF / archive_rel
    assert archive_path.is_file(), f'COMPRESSED_SURFACE_MISSING:{archive_rel}'
    SURFACE_JSON = MATERIALIZED / 'qualified_rigging_surface.json'
    with gzip.open(archive_path, 'rb') as src, SURFACE_JSON.open('wb') as dst:
        shutil.copyfileobj(src, dst, length=1<<20)
    assert SURFACE_JSON.stat().st_size == int(surface_transport['uncompressed_bytes']), 'SURFACE_UNCOMPRESSED_SIZE_DRIFT'
    assert sha256(SURFACE_JSON) == str(surface_transport['uncompressed_sha256']), 'SURFACE_UNCOMPRESSED_SHA_DRIFT'
else:
    SURFACE_JSON = BASE_HANDOFF / 'qualified_rigging_surface.json'
    assert SURFACE_JSON.is_file(), f'SURFACE_JSON_MISSING:{SURFACE_JSON}'

# Pull only the already-CI-verified exact code commit; do not follow moving branch HEAD.
subprocess.run(['git','init',str(SRC)], check=True)
subprocess.run(['git','-C',str(SRC),'remote','add','origin','https://github.com/merynz/RealSaS-OPT.git'], check=True)
subprocess.run(['git','-C',str(SRC),'fetch','--depth=1','origin',EXECUTION_COMMIT], check=True)
subprocess.run(['git','-C',str(SRC),'checkout','--detach','FETCH_HEAD'], check=True)
head=subprocess.check_output(['git','-C',str(SRC),'rev-parse','HEAD'], text=True).strip()
assert head==EXECUTION_COMMIT, f'EXECUTION_COMMIT_DRIFT:{head}'

runner=SRC/'tools/training/run_knight_geppetto_demo_fit_v3_mechanical.py'
builder=SRC/'experiments/geppetto_reference_strength_fullstack_v1/mechanically_meaningful_target_v2.py'
assert runner.is_file() and builder.is_file()
assert sha256(builder)==sha256(V2_HANDOFF/'mechanically_meaningful_target_v2.py'), 'DRIVE_REPO_TARGET_BUILDER_DRIFT'
subprocess.run([sys.executable,'-m','py_compile',str(builder),str(runner)],check=True)
subprocess.run([sys.executable,'-m','pip','install','--no-input','--progress-bar','off','-r',str(SRC/'requirements/mainline-ci.txt')],check=True)
subprocess.run([sys.executable,'-m','pytest','-q',str(SRC/'tests/geppetto/test_mechanically_meaningful_target_v2.py')],cwd=SRC,check=True)
print('EXECUTION_SOURCE_READY', head)


In [ ]:
cmd=[
    sys.executable, str(SRC/'tools/training/run_knight_geppetto_demo_fit_v3_mechanical.py'),
    '--surface-json', str(SURFACE_JSON),
    '--surface-qualification-json', str(BASE_HANDOFF/'rigging_surface_qualification.json'),
    '--teacher-target-npz', str(V2_HANDOFF/'teacher_target_v2.npz'),
    '--teacher-target-report', str(V2_HANDOFF/'teacher_target_v2_report.json'),
    '--target-policy', str(V2_HANDOFF/'GEPPETTO_MECHANICALLY_MEANINGFUL_RIG_POLICY_V1_20260925.json'),
    '--preregistration-ir', str(BASE_HANDOFF/'model_fit_preregistration.json'),
    '--experiment-prereg', str(V2_HANDOFF/'experiment_preregistration_v2.json'),
    '--model-source', str(BASE_HANDOFF/'model_source.py'),
    '--camera-set-json', str(BASE_HANDOFF/'qualified_camera_set.json'),
    '--observation-dir', str(BASE_HANDOFF/'observations'),
    '--output-dir', str(LOCAL_OUTPUT),
]
env=dict(os.environ)
env['PYTHONPATH']=str(SRC)+((':'+env['PYTHONPATH']) if env.get('PYTHONPATH') else '')
subprocess.run(cmd,cwd=SRC,env=env,check=True)


In [ ]:
out_manifest_path=LOCAL_OUTPUT/'GEPPETTO_KNIGHT_OUTPUT_MANIFEST.json'
assert out_manifest_path.is_file(), 'OUTPUT_MANIFEST_MISSING'
out_manifest=json.loads(out_manifest_path.read_text(encoding='utf-8'))
assert out_manifest['schema']=='RealSaS.KnightGeppettoFitOutputManifest.v1', out_manifest
assert out_manifest['status']=='PASS', out_manifest
assert out_manifest['product_authority_claimed'] is False, out_manifest
for role, meta in out_manifest['files'].items():
    p=LOCAL_OUTPUT/meta['path']
    assert p.is_file(), f'OUTPUT_FILE_MISSING:{role}:{p}'
    assert sha256(p)==meta['sha256'], f'OUTPUT_SHA_DRIFT:{role}'

result=json.loads((LOCAL_OUTPUT/'GEPPETTO_KNIGHT_RESULT.json').read_text(encoding='utf-8'))
assert result['schema']=='RealSaS.KnightGeppettoDemoFitMechanicalRig.v3', result
assert result['status']=='TERMINAL_PASS', result
assert result['all_diffusion_seeds_passed'] is True, result
assert int(result['required_terminal_checks'])==REQUIRED_CONSECUTIVE_FULL_PASSES, result
assert int(result['terminal_streak'])>=REQUIRED_CONSECUTIVE_FULL_PASSES, result
assert int(result['target_count'])==EXPECTED_TARGET_COUNT, result
assert result['target_role_counts']==EXPECTED_ROLE_COUNTS, result
assert result['target_rule']==EXPECTED_TARGET_RULE, result
assert result['teacher_feedback_during_free_running_inference'] is False, result
assert result['teacher_inference_inputs_used'] is False, result
assert result['historical_checkpoint_loaded'] is False, result
assert result['fresh_from_scratch'] is True, result
assert result['product_authority_claimed'] is False, result
assert result['generalization_claimed'] is False, result

# The terminal claim must be backed by 3 consecutive full checks, not one lucky check.
trace=list(result.get('trace') or [])
assert len(trace)>=REQUIRED_CONSECUTIVE_FULL_PASSES, 'TRACE_TOO_SHORT_FOR_3_PASS_RULE'
terminal_trace=trace[-REQUIRED_CONSECUTIVE_FULL_PASSES:]
assert [int(x['terminal_streak']) for x in terminal_trace]==[1,2,3], terminal_trace
steps=[int(x['step']) for x in terminal_trace]
assert all((steps[i+1]-steps[i])==CHECK_EVERY for i in range(len(steps)-1)), steps
for check in terminal_trace:
    assert check['pass'] is True, check
    seeds=list(check.get('diffusion_seed_reports') or [])
    assert [int(s['seed']) for s in seeds]==DIFFUSION_SEEDS, seeds
    assert all(s['pass'] is True for s in seeds), seeds
    assert all(float(s['root_accuracy'])==1.0 for s in seeds), seeds
    assert all(float(s['parent_accuracy'])==1.0 for s in seeds), seeds

runtime=dict(result.get('runtime_environment') or {})
assert runtime.get('cuda_available') is True, runtime
gpu_name=str((runtime.get('gpu0') or {}).get('name') or '')
assert 'A100' in gpu_name, runtime
assert runtime.get('torch_version') and runtime.get('torch_cuda_runtime'), runtime

visual=LOCAL_OUTPUT/'visual_evidence'/'KNIGHT_GEPPETTO_8VIEW_CONTACT_SHEET.png'
assert visual.is_file(), 'CONTACT_SHEET_MISSING'

if DRIVE_OUTPUT.exists():
    shutil.rmtree(DRIVE_OUTPUT)
shutil.copytree(LOCAL_OUTPUT, DRIVE_OUTPUT)
marker={
    'schema':'RealSaS.KnightGeppettoMechanicallyMeaningfulV2RunAllComplete.v1',
    'status':'PASS',
    'execution_repo_commit':EXECUTION_COMMIT,
    'self_hosted_ci_run_id':EXPECTED_CI_RUN_ID,
    'required_consecutive_full_passes':REQUIRED_CONSECUTIVE_FULL_PASSES,
    'closure_step':int(result['closure_step']),
    'terminal_check_steps':steps,
    'target_count':EXPECTED_TARGET_COUNT,
    'target_role_counts':EXPECTED_ROLE_COUNTS,
    'target_rule':EXPECTED_TARGET_RULE,
    'base_handoff_manifest_sha256':sha256(base_manifest_path),
    'v2_handoff_manifest_sha256':sha256(v2_manifest_path),
    'output_manifest_sha256':sha256(DRIVE_OUTPUT/'GEPPETTO_KNIGHT_OUTPUT_MANIFEST.json'),
    'contact_sheet_sha256':sha256(DRIVE_OUTPUT/'visual_evidence'/'KNIGHT_GEPPETTO_8VIEW_CONTACT_SHEET.png'),
    'product_authority_claimed':False,
    'generalization_claimed':False,
}
(DRIVE_OUTPUT/'RUN_ALL_COMPLETE.json').write_text(json.dumps(marker,indent=2,sort_keys=True)+'\n',encoding='utf-8')
print('GEPPETTO_MECHANICALLY_MEANINGFUL_V2_RUN_ALL_PASS')
print(json.dumps(marker,sort_keys=True))
